In [ ]:
import logfire

from typing import Any
from rich.pretty import pprint

from pydantic_ai.tools import RunContext
from pydantic_ai.mcp import MCPServerStreamableHTTP, CallToolFunc, ToolResult

from por.llm_agents import NietzscheAdvisor, NietzscheAdvisorDeps

In [ ]:
logfire.configure(service_name="por")
_ = logfire.instrument_mcp()
_ = logfire.instrument_pydantic_ai()

In [ ]:
async def process_tool_call(
    ctx: RunContext[int],
    call_tool: CallToolFunc,
    tool_name: str,
    args: dict[str, Any],
) -> ToolResult:
    call_result = await call_tool(
        tool_name,
        args,
        metadata={"deps": ctx.deps},
    )

    pprint(
        {
            "tool": tool_name,
            "args": args,
            "result": call_result,
        }
    )

    return call_result

In [ ]:
mcp = MCPServerStreamableHTTP(
    url="http://localhost:8000/mcp",
    process_tool_call=process_tool_call,
)

In [ ]:
nietzsche_advisor = NietzscheAdvisor(mcp_servers=[mcp])
async with nietzsche_advisor.agent.run_mcp_servers():
    nietzsche_advisor_output = await nietzsche_advisor.generate(
        user_prompt="Deliver piercing, symbolic, and transformative insight.",
        agent_deps=NietzscheAdvisorDeps(
            collection="nietzsche",
            psychological_profile="Unaviable",
            question="Que debo hacer para convertirme en un León?",
            output_language="Spanish",
        ),
    )

In [ ]:
pprint(nietzsche_advisor_output)